# 작은 Transformer 만들기 — 수식과 PyTorch 구현

문자 단위로 **다음 글자를 예측하는 2층 decoder-only Transformer**를 처음부터 구성한다.
한국어 예제 문장을 사용하며, 사전 학습 모델이나 외부 데이터 다운로드 없이 구조와 학습 흐름을 살펴본다.

예: `오늘은 맑` → 다음 문자 `은`을 예측한다. 학습 후에는 다음 문자를 반복해서 생성한다.

| 항목 | 기본값 |
| --- | --- |
| 토큰 | 문자 1개 |
| 최대 문맥 길이 $T$ | 64 |
| 은닉 차원 $d$ | 64 |
| Attention head 수 $h$ | 4 |
| Transformer block 수 $L$ | 2 |
| FFN 내부 차원 | $4d = 256$ |
| 배치 크기 $B$ | 32 |
| 학습 step 수 | 300 |

**실행 원칙:** 이 노트북의 코드 셀은 RunPod GPU에서만 실행한다. 현재 파일에는 실행 결과가 없다.
장난감 데이터의 loss는 구조와 학습 동작을 관찰하기 위한 값이며, 실제 한국어 능력을 평가하는 지표가 아니다.

## 1. RunPod에서 실행 준비

1. CUDA용 PyTorch와 JupyterLab이 있는 RunPod GPU Pod를 준비한다.
2. 이 파일을 Pod의 `/workspace/flownote-agent/research/`에 업로드한다.
3. Pod의 **Connect → Jupyter Lab**에서 파일을 열고 PyTorch가 설치된 Python 커널을 선택한다.
4. 아래 셀부터 순서대로 실행한다. CPU나 로컬 환경으로 자동 전환하지 않는다.

추가 패키지가 없다면 **RunPod의 Jupyter 터미널에서만** 다음을 실행한다. 이미지에 있는 CUDA PyTorch는 그대로 사용한다.

```bash
python -m pip install matplotlib
```

노트북의 첫 셀은 `RUNPOD_POD_ID`, `/workspace`, CUDA 사용 가능 여부를 확인한다.
체크포인트와 실험 기록은 `/workspace/flownote-agent/research/artifacts/` 아래에 실행별로 저장한다.

접속 방법: [RunPod 공식 문서](https://docs.runpod.io/pods/connect-to-a-pod).

In [ ]:
import os
from pathlib import Path

if not os.environ.get("RUNPOD_POD_ID") or not Path("/workspace").is_dir():
    raise RuntimeError("이 노트북은 RunPod의 /workspace 환경에서만 실행하세요.")

import json
import math
import random
import time
from dataclasses import asdict, dataclass
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F


def require_runpod_gpu():
    if not os.environ.get("RUNPOD_POD_ID") or not Path("/workspace").is_dir():
        raise RuntimeError("RunPod 환경이 필요합니다. 로컬 실행을 중단합니다.")
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA GPU가 필요합니다. RunPod GPU와 PyTorch 커널을 확인하세요.")


require_runpod_gpu()
device = torch.device("cuda")
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
# 같은 seed라도 GPU 종류와 라이브러리 버전에 따라 결과가 조금 달라질 수 있다.
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))

## 2. 데이터를 숫자로 바꾸기

문자 집합의 크기를 $V$라고 할 때, 각 문자에 $0, \ldots, V-1$의 정수 ID를 부여한다.

시퀀스 $[c_0,c_1,\ldots,c_T]$에서 입력과 정답은 한 칸 어긋난다.

$$
x=[c_0,c_1,\ldots,c_{T-1}],\qquad
 y=[c_1,c_2,\ldots,c_T].
$$

아래에서는 서로 다른 600개 문장을 생성하고 **문장 단위로 먼저** 학습 90%, 검증 10%로 나눈다.
그 뒤 각 집합을 별도 문자 스트림으로 연결한다. 같은 원문 문장이 양쪽에 들어가지 않는다.
문장 사이의 줄바꿈도 토큰이며, batch의 문맥은 문장 경계를 넘어갈 수 있다.
검증 문장은 학습 문장과 어휘·문법을 공유하므로 일반적인 언어 모델 평가보다 쉬운 예제다.

In [ ]:
@dataclass(frozen=True)
class Config:
    block_size: int = 64
    d_model: int = 64
    n_heads: int = 4
    n_layers: int = 2
    dropout: float = 0.1
    batch_size: int = 32
    learning_rate: float = 3e-4
    weight_decay: float = 0.01
    max_steps: int = 300
    eval_interval: int = 50
    eval_batches: int = 10


cfg = Config()
assert cfg.d_model % cfg.n_heads == 0

subjects = ["나는", "우리는", "친구는", "학생은", "선생님은"]
places = ["집에서", "학교에서", "도서관에서", "공원에서", "교실에서"]
activities = ["책을 읽는다", "글을 쓴다", "수학을 공부한다", "음악을 듣는다", "모델을 만든다", "코드를 작성한다"]
weather = ["맑은", "흐린", "따뜻한", "시원한"]
sentences = [
    f"오늘은 {w} 날이다. {s} {p} {a}.\n"
    for w in weather for s in subjects for p in places for a in activities
]
rng = random.Random(SEED)
rng.shuffle(sentences)
split_at = int(len(sentences) * 0.9)
train_sentences = sentences[:split_at]
val_sentences = sentences[split_at:]
assert not set(train_sentences) & set(val_sentences)
train_text = "".join(train_sentences)
val_text = "".join(val_sentences)

# 어휘만 전체 예제로 정의한다. 검증 문장의 순서는 학습에 사용하지 않는다.
chars = sorted(set(train_text + val_text))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
vocab_size = len(chars)


def encode(text):
    unknown = sorted(set(text) - set(stoi))
    if unknown:
        raise ValueError(f"예제 어휘에 없는 문자가 있습니다: {unknown!r}")
    return [stoi[ch] for ch in text]


def decode(ids):
    return "".join(itos[int(i)] for i in ids)


require_runpod_gpu()
# 작은 예제이므로 두 스트림을 GPU에 모두 올린다.
train_data = torch.tensor(encode(train_text), dtype=torch.long, device=device)
val_data = torch.tensor(encode(val_text), dtype=torch.long, device=device)
for data in (train_data, val_data):
    assert len(data) >= cfg.block_size + 1

train_rng = torch.Generator(device=device).manual_seed(SEED + 1)


def get_batch(split, generator):
    if split not in {"train", "val"}:
        raise ValueError("split은 train 또는 val이어야 합니다.")
    data = train_data if split == "train" else val_data
    starts = torch.randint(
        len(data) - cfg.block_size, (cfg.batch_size,),
        device=device, generator=generator,
    )
    positions = starts[:, None] + torch.arange(cfg.block_size, device=device)
    return data[positions], data[positions + 1]


x_demo, y_demo = get_batch("train", train_rng)
print("문장 수:", len(train_sentences), "train /", len(val_sentences), "val")
print("어휘 크기:", vocab_size)
print("입력:", repr(decode(x_demo[0].tolist())))
print("정답:", repr(decode(y_demo[0].tolist())))
print("batch shape:", tuple(x_demo.shape))

## 3. Token embedding과 위치 embedding

문자 ID는 크기나 의미가 있는 실수가 아니라 인덱스다. 학습 가능한 표에서 벡터를 꺼내고,
각 위치의 학습 가능한 벡터를 더한다.

$$
H^{(0)}_{b,t}=E_{\mathrm{token}}[x_{b,t}]+E_{\mathrm{pos}}[t].
$$

$E_{\mathrm{token}}\in\mathbb{R}^{V\times d}$,
$E_{\mathrm{pos}}\in\mathbb{R}^{T_{\max}\times d}$,
$H^{(0)}\in\mathbb{R}^{B\times T\times d}$이다.
이 노트북은 sinusoidal 위치 encoding 대신 학습 가능한 위치 embedding을 사용한다.

## 4. Causal multi-head self-attention

각 head의 차원은 $d_k=d/h$이다. 하나의 선형 층으로 Query, Key, Value를 만든 뒤 head로 분할한다.
편향 항은 수식에서 생략했지만 코드에서는 사용한다.

$$
Q=HW_Q,\qquad K=HW_K,\qquad V=HW_V.
$$

각 head에서:

$$
S_{ij}=\frac{q_i^\top k_j}{\sqrt{d_k}}+M_{ij},\qquad
M_{ij}=\begin{cases}0&j\le i\\-\infty&j>i\end{cases},
$$

$$
A=\operatorname{softmax}(S),\qquad O=AV.
$$

Softmax는 key 축, 즉 마지막 축에 적용한다. $M$은 미래 문자를 보지 못하게 한다.
여러 head의 출력을 연결하고 다시 $d$차원으로 투영한다.

$$
\operatorname{MHA}(H)=\operatorname{Concat}(O_1,\ldots,O_h)W_O.
$$

아래 구현은 수식에 대응하는 행렬곱, 마스크, softmax를 직접 보여준다.
학습 시에는 attention 확률과 출력에 dropout을 적용하며, 평가 시에는 비활성화한다.
[Attention Is All You Need](https://arxiv.org/abs/1706.03762)의 attention 구조를 바탕으로 한다.

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_heads = cfg.n_heads
        self.head_dim = cfg.d_model // cfg.n_heads
        self.qkv = nn.Linear(cfg.d_model, 3 * cfg.d_model)
        self.proj = nn.Linear(cfg.d_model, cfg.d_model)
        self.attn_dropout = nn.Dropout(cfg.dropout)
        self.out_dropout = nn.Dropout(cfg.dropout)
        mask = torch.tril(torch.ones(cfg.block_size, cfg.block_size, dtype=torch.bool))
        self.register_buffer("causal_mask", mask, persistent=False)

    def forward(self, x):
        B, T, D = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        # (B, T, D) -> (B, h, T, d_k)
        q = q.reshape(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.reshape(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.reshape(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        scores = scores.masked_fill(~self.causal_mask[:T, :T], float("-inf"))
        attention = F.softmax(scores, dim=-1)
        out = self.attn_dropout(attention) @ v
        out = out.transpose(1, 2).contiguous().reshape(B, T, D)
        return self.out_dropout(self.proj(out))

## 5. Feed-forward network와 residual 연결

각 위치에 같은 FFN을 독립적으로 적용한다. 이 예제는 GELU 활성화 함수와 **Pre-LayerNorm**을 사용한다.

$$
\operatorname{FFN}(u)=\operatorname{GELU}(uW_1+b_1)W_2+b_2,
$$

$$
\widetilde H=H+\operatorname{MHA}(\operatorname{LN}(H)),\qquad
H'=\widetilde H+\operatorname{FFN}(\operatorname{LN}(\widetilde H)).
$$

$W_1\in\mathbb{R}^{d\times4d}$, $W_2\in\mathbb{R}^{4d\times d}$이다.
LayerNorm은 각 토큰의 마지막 $d$차원에 적용하고, residual 연결은 입력을 출력에 더한다.
Dropout은 위 수식에서 생략했으며 코드의 FFN 출력에도 적용한다.
원 논문의 전체 encoder-decoder 모델과 달리 여기서는 causal decoder block만 쌓는다.

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attention = CausalSelfAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = nn.Sequential(
            nn.Linear(cfg.d_model, 4 * cfg.d_model),
            nn.GELU(),
            nn.Linear(4 * cfg.d_model, cfg.d_model),
            nn.Dropout(cfg.dropout),
        )

    def forward(self, x):
        x = x + self.attention(self.ln1(x))
        return x + self.ffn(self.ln2(x))

## 6. 출력 확률과 cross-entropy loss

$L$개 block의 결과를 정규화하고 어휘 크기 $V$로 투영한다.

$$
Z=\operatorname{LN}(H^{(L)})W_{\mathrm{vocab}}+b_{\mathrm{vocab}},\qquad
p_{b,t}=\operatorname{softmax}(Z_{b,t}).
$$

Logit $Z$의 shape은 $(B,T,V)$이다. 학습 목표는 실제 다음 문자의 음의 로그 확률 평균을 줄이는 것이다.

$$
\mathcal L=-\frac{1}{BT}\sum_{b=1}^{B}\sum_{t=1}^{T}\log p_{b,t,y_{b,t}}.
$$

PyTorch의 `cross_entropy`에는 **softmax 전 logits**를 넣는다. 내부에서 log-softmax와 loss를 계산한다.
[PyTorch cross_entropy 문서](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.cross_entropy.html).

In [ ]:
class TinyTransformerLM(nn.Module):
    def __init__(self, vocab_size, cfg):
        super().__init__()
        self.cfg = cfg
        self.token_embedding = nn.Embedding(vocab_size, cfg.d_model)
        self.position_embedding = nn.Embedding(cfg.block_size, cfg.d_model)
        self.blocks = nn.Sequential(*[TransformerBlock(cfg) for _ in range(cfg.n_layers)])
        self.final_norm = nn.LayerNorm(cfg.d_model)
        self.lm_head = nn.Linear(cfg.d_model, vocab_size)
        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, ids, targets=None):
        if ids.ndim != 2:
            raise ValueError("입력 shape은 (B, T)여야 합니다.")
        _, T = ids.shape
        if not 1 <= T <= self.cfg.block_size:
            raise ValueError("문맥 길이는 1 이상 block_size 이하여야 합니다.")
        positions = torch.arange(T, device=ids.device)
        x = self.token_embedding(ids) + self.position_embedding(positions)[None, :, :]
        logits = self.lm_head(self.final_norm(self.blocks(x)))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss


require_runpod_gpu()
model = TinyTransformerLM(vocab_size, cfg).to(device)
parameter_count = sum(p.numel() for p in model.parameters())
print(f"파라미터 수: {parameter_count:,}")
print(model)

## 7. 학습 전에 shape과 미래 정보 차단 확인

뒤쪽 문자만 바꾸었을 때 앞쪽 위치의 logits가 같아야 한다.
Dropout의 무작위 영향을 제거하기 위해 `eval()`에서 비교한다.
이 검사는 RunPod에서 노트북 실행 시 수행된다.

In [ ]:
require_runpod_gpu()
model.eval()
with torch.no_grad():
    logits, loss = model(x_demo, y_demo)
    assert logits.shape == (cfg.batch_size, cfg.block_size, vocab_size)
    assert loss.ndim == 0 and torch.isfinite(loss).item()

    prefix_length = cfg.block_size // 2
    changed = x_demo.clone()
    changed[:, prefix_length:] = (changed[:, prefix_length:] + 1) % vocab_size
    changed_logits, _ = model(changed)
    torch.testing.assert_close(
        logits[:, :prefix_length], changed_logits[:, :prefix_length],
        rtol=1e-5, atol=1e-6,
    )
model.train()
print("shape, 유한한 loss, causal mask 검사 통과")

## 8. 학습과 검증

`loss.backward()`로 $\nabla_\theta\mathcal L$을 계산하고 AdamW로 파라미터를 갱신한다.
Gradient clipping은 전체 gradient norm이 1을 넘으면 크기를 줄인다.

검증에서는 `eval()`과 `no_grad()`를 사용한다. 평가용 무작위 생성기를 별도로 초기화하여
매번 같은 구간에서 loss를 측정하고 학습 batch 추출에 영향을 주지 않는다.
검증 loss는 전체 검증 스트림의 정확한 평균이 아니라 고정된 무작위 batch의 평균 추정치다.

300 step은 초기 실험 설정이다. **이 셀을 다시 실행하면 현재 모델을 추가로 300 step 학습한다.**
처음부터 다시 학습하려면 커널을 재시작하고 위에서부터 실행한다.

In [ ]:
@torch.no_grad()
def estimate_loss():
    was_training = model.training
    model.eval()
    result = {}
    try:
        for offset, split in enumerate(("train", "val")):
            eval_rng = torch.Generator(device=device).manual_seed(SEED + 100 + offset)
            losses = []
            for _ in range(cfg.eval_batches):
                x, y = get_batch(split, eval_rng)
                _, loss = model(x, y)
                losses.append(loss.detach())
            result[split] = torch.stack(losses).mean().item()
    finally:
        model.train(was_training)
    return result


require_runpod_gpu()
optimizer = torch.optim.AdamW(
    model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay,
)
history = []
initial = estimate_loss()
history.append({"step": 0, **initial})
print(f"step    0 | train {initial['train']:.4f} | val {initial['val']:.4f}")
model.train()
torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()
start_time = time.perf_counter()

for step in range(1, cfg.max_steps + 1):
    x, y = get_batch("train", train_rng)
    optimizer.zero_grad(set_to_none=True)
    _, loss = model(x, y)
    if not torch.isfinite(loss).item():
        raise RuntimeError(f"step {step}: loss가 유한하지 않습니다.")
    loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0, error_if_nonfinite=True)
    optimizer.step()
    if step % cfg.eval_interval == 0 or step == cfg.max_steps:
        metrics = estimate_loss()
        history.append({"step": step, **metrics})
        print(f"step {step:4d} | train {metrics['train']:.4f} | val {metrics['val']:.4f}")

torch.cuda.synchronize()
elapsed_seconds = time.perf_counter() - start_time
peak_memory_mib = torch.cuda.max_memory_allocated() / 1024**2
print(f"소요 시간: {elapsed_seconds:.1f}초 / 최대 PyTorch 할당 메모리: {peak_memory_mib:.1f} MiB")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot([m["step"] for m in history], [m["train"] for m in history], marker="o", label="train")
ax.plot([m["step"] for m in history], [m["val"] for m in history], marker="o", label="validation")
ax.set(xlabel="Optimizer step", ylabel="Cross-entropy (nats)", title="Tiny Transformer learning curve")
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()

## 9. 다음 문자를 반복해서 생성

마지막 위치의 logits $z$에 temperature $\tau>0$를 적용하고 다음 문자 하나를 샘플링한다.

$$
p_i=\frac{\exp(z_i/\tau)}{\sum_j\exp(z_j/\tau)},\qquad c_{\mathrm{next}}\sim\operatorname{Categorical}(p).
$$

작은 $\tau$는 높은 logit의 문자에 확률을 더 집중시킨다.
문맥 길이가 64를 넘으면 최근 64개 문자만 입력하고 위치 인덱스를 다시 0부터 부여한다.
이 예제에는 별도 종료 토큰이 없어서 지정한 문자 수까지 생성한다.

In [ ]:
@torch.no_grad()
def generate(model, prompt="오늘은 ", max_new_tokens=120, temperature=0.8, seed=123):
    require_runpod_gpu()
    if not prompt:
        raise ValueError("prompt는 비어 있으면 안 됩니다.")
    if not math.isfinite(temperature) or temperature <= 0:
        raise ValueError("temperature는 유한한 양수여야 합니다.")
    if max_new_tokens < 0:
        raise ValueError("max_new_tokens는 0 이상이어야 합니다.")
    model_device = next(model.parameters()).device
    ids = torch.tensor([encode(prompt)], dtype=torch.long, device=model_device)
    sample_rng = torch.Generator(device=model_device).manual_seed(seed)
    was_training = model.training
    model.eval()
    try:
        for _ in range(max_new_tokens):
            context = ids[:, -model.cfg.block_size:]
            logits, _ = model(context)
            probs = F.softmax(logits[:, -1, :] / temperature, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1, generator=sample_rng)
            ids = torch.cat((ids, next_id), dim=1)
    finally:
        model.train(was_training)
    return decode(ids[0].tolist())


generated_text = generate(model)
print(generated_text)

## 10. 체크포인트 저장과 재로딩 확인

가중치, optimizer 상태, 문자 사전, 모델 설정과 seed를 저장한다.
실행 시간·GPU·패키지 버전·loss·생성 결과도 함께 기록한다.

저장 후 새 모델에 가중치를 읽어 같은 입력의 logits가 복원되는지 확인한다.
이 체크포인트에는 난수 생성기 상태와 데이터 커서가 없으므로 **학습을 완전히 동일하게 이어가는 기능은 포함하지 않는다.**

In [ ]:
require_runpod_gpu()
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
run_dir = Path("/workspace/flownote-agent/research/artifacts") / f"tiny-transformer-{run_id}"
run_dir.mkdir(parents=True, exist_ok=False)
checkpoint_path = run_dir / "model.pt"
# 가중치는 CPU tensor로 저장하여 파일의 GPU 장치 의존성을 줄인다.
torch.save({
    "model_state": {name: tensor.detach().cpu() for name, tensor in model.state_dict().items()},
    "optimizer_state": optimizer.state_dict(),
    "config": asdict(cfg),
    "chars": chars,
    "seed": SEED,
    "completed_steps": cfg.max_steps,
}, checkpoint_path)

checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
restored_cfg = Config(**checkpoint["config"])
restored_model = TinyTransformerLM(len(checkpoint["chars"]), restored_cfg).to(device)
restored_model.load_state_dict(checkpoint["model_state"])
model.eval()
restored_model.eval()
with torch.no_grad():
    original_logits, _ = model(x_demo[:1])
    restored_logits, _ = restored_model(x_demo[:1])
    torch.testing.assert_close(original_logits, restored_logits, rtol=1e-5, atol=1e-6)

report = {
    "run_id": run_id,
    "config": asdict(cfg),
    "seed": SEED,
    "torch_version": str(torch.__version__),
    "cuda_version": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0),
    "parameter_count": parameter_count,
    "vocab_size": vocab_size,
    "train_sentences": len(train_sentences),
    "val_sentences": len(val_sentences),
    "elapsed_seconds": elapsed_seconds,
    "peak_memory_mib": peak_memory_mib,
    "history": history,
    "generated_text": generated_text,
    "reload_check_passed": True,
}
(run_dir / "metrics.json").write_text(json.dumps(report, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
(run_dir / "sample.txt").write_text(generated_text + "\n", encoding="utf-8")
fig.savefig(run_dir / "loss.png", dpi=150, bbox_inches="tight")
print("저장 및 재로딩 검사 통과:", run_dir)

## 11. 결과를 읽는 방법과 다음 실험

- 학습과 검증 loss가 초기값보다 낮아지는지 살펴본다. 초기값을 바꾸거나 학습 시간을 늘리면 결과가 달라질 수 있다.
- 검증 loss가 상승하는데 학습 loss만 계속 내려가면 과적합 가능성을 살펴본다.
- `n_layers`, `d_model`, `block_size` 중 하나씩 바꿔 loss와 GPU 메모리를 비교한다.
- 구조를 이해한 뒤 실제 문서 데이터, subword tokenizer, 효율적인 attention 구현으로 확장한다.

이 파일은 아직 실행되지 않았다. 실제 수치와 생성 예제는 RunPod에서 실행한 뒤 기록한다.

참고 자료:
- [Attention Is All You Need](https://arxiv.org/abs/1706.03762)
- [PyTorch cross_entropy](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.cross_entropy.html)
- [RunPod 연결 및 JupyterLab 안내](https://docs.runpod.io/pods/connect-to-a-pod)